In [1]:
%load_ext autoreload
%autoreload 2
from sindex.sources.openalex.snapshot import (
    run_openalex_sweep, 
    process_openalex_citations_for_datasets_year,
    export_oa_topics_to_ndjson,
    export_citations_to_ndjson_year
)
import duckdb
import os

## Converting OA snapshot file from .gz to parquet files 

This needs to be run every time new OpenAlex snapshot folders are downloaded. This is necessary for the next step of the process with DuckDB.

In [2]:
# Create parquet files from oa works gz files
source_dir = r"D:\may-2026-data\external\openalex-snapshot\data\works"
meta_out = r"D:\may-2026-data\external\openalex-snapshot\parquets\metadata"
cite_out = r"D:\may-2026-data\external\openalex-snapshot\parquets\citations"

In [3]:
results = run_openalex_sweep(source_dir, meta_out, cite_out, max_workers=16)

if results['errors']:
    print(f"\nCompleted with errors: {len(results['errors'])}")

Extracting from 627 files
Progress: 627/627 (New: 627, Skipped: 0)

## Topics and citations for DOIs

### Get topics for datasets of interest

In [3]:
oa_topics_citation_db_path = r"D:\combined-data\external\openalex-snapshot\openalex_topics_citations.db"
dataset_db_path = r"D:\combined-data\records\slim-records\datacite-slim-records.duckdb"
meta_folders = [
        r"D:\may-2026-data\external\openalex-snapshot\parquets\metadata",  # new, scanned first
        r"D:\pipeline-data\external\openalex-snapshot\parquets\metadata",  # old
    ]
new_datasets_since="2026-05-01"

In [14]:
process_openalex_topics_for_datasets(
    db_path=oa_topics_citation_db_path,
    dataset_db_path=dataset_db_path,
    meta_folders=meta_folders,
    new_datasets_since=new_datasets_since,
    mem_limit="32GB",
    reset_file_tracking_table = True
)

Resetting file tracking table, existing topics will be preserved...
Filtering to datasets added since 2026-05-01
Starting scan of 2343 metadata files...
Scanned: 2343/2343 | Elapsed: 28:52
Done! Unique entries: 63,490,876 | Total Time: 148.87 min


In [20]:
export_oa_topics_to_ndjson(
    db_path=r"D:\combined-data\external\openalex-snapshot\openalex_topics_citations.db",
    output_path=r"D:\may-2026-data\topics\doi_topics_oa.ndjson",
    since_date="2026-05-01"
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Export completed: D:\may-2026-data\topics\doi_topics_oa.ndjson
Total rows written: 4,749,534


### Get citations

In [5]:
process_openalex_citations_for_datasets_year(
    db_path=oa_topics_citation_db_path,
    dataset_db_path=dataset_db_path,
    new_cite_folders=[
        r"D:\may-2026-data\external\openalex-snapshot\parquets\citations",
    ],
    old_cite_folders=[
        r"D:\pipeline-data\external\openalex-snapshot\parquets\citations",
    ],
    meta_folders=meta_folders,
    new_datasets_since="2026-05-01",
    mem_limit="32GB",
    reset_step1 = True
)

Resetting Step 1 tables, existing citations will be preserved...
Step 1 - Pass 1: Scanning new citation parquets for all datasets
Pass 1: Finding citations in 627 files
 > Progress: 627/627 | Elapsed: 00:33:05
Step 1 - Pass 2: Scanning old citation parquets for datasets added since 2026-05-01
Pass 2: Finding citations in 1716 files
 > Progress: 1716/1716 | Elapsed: 01:05:57

Step 2: Mapping citing IDs and joining pubyear
Done! Newly added citations: 37,216, Total in table: 2,792,190, Total Time: 66.75 min


In [10]:
export_citations_to_ndjson_year(
    db_path=oa_topics_citation_db_path,
    out_ndjson = r"D:\may-2026-data\citations\openalex\oa_citations.ndjson",
    since_date = "2026-05-01"
)

Starting export to D:\may-2026-data\citations\openalex\oa_citations.ndjson
Processed: 37,235 | Elapsed: 2.31s
Complete. Total citation records: 37,235


## ----- EXTRA ----- 

## Analysis

### Topics

In [34]:
db_path = r"I:\pipeline-data\openalex_processed\openalex_topics_citations.db"

In [26]:
con = duckdb.connect(db_path)
count = con.execute("""
    SELECT COUNT(*) 
    FROM my_datasets_topics 
    WHERE topic_score > 0.5
""").fetchone()[0]

print(f"Number of items with a high topic score: {count:,}")
con.close()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Number of items with a high topic score: 6,618,256


In [36]:
con = duckdb.connect(db_path)
display(con.execute("""
    SELECT*
    FROM my_datasets_topics 
    LIMIT 5
""").df())

con.close()

,oa_id,doi,publication_date,created_date,topic_id,topic_name,topic_score,best_date
0,W6895049865,10.57451/lhd.a.ha3.165478.1,2021-09-02T00:00:00,2025-06-10T08:55:36+00:00,None,None,NaN,2021-09-02T00:00:00
1,W6932788472,10.57451/lhd.a.ha3.165644.1,2021-09-02T00:00:00,2025-06-10T08:55:55+00:00,None,None,NaN,2021-09-02T00:00:00
2,W6913700087,10.57451/lhd.a.ha3.165980.1,2021-09-02T00:00:00,2025-06-10T08:56:35+00:00,None,None,NaN,2021-09-02T00:00:00
3,W6932605310,10.57451/lhd.a.ha3.166099.1,2021-09-02T00:00:00,2025-06-10T08:56:47+00:00,None,None,NaN,2021-09-02T00:00:00
4,W6932634202,10.57451/lhd.a.ha3.166266.1,2021-09-02T00:00:00,2025-06-10T08:57:07+00:00,None,None,NaN,2021-09-02T00:00:00


In [29]:
con = duckdb.connect(db_path)
query = """
    SELECT 
        COUNT(DISTINCT oa_id) AS oa_id,
        COUNT(DISTINCT doi) AS doi,
        COUNT(DISTINCT publication_date) AS publication_date,
        COUNT(DISTINCT created_date) AS created_date,
        COUNT(DISTINCT topic_id) AS topic_id,
        COUNT(DISTINCT topic_name) AS topic_name,
        COUNT(DISTINCT topic_score) AS topic_score,
        COUNT(DISTINCT best_date) AS best_date
    FROM my_datasets_topics
"""

unique_counts_df = con.execute(query).df()
display(unique_counts_df)
con.close()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

NameError: name 'df' is not defined

In [30]:
display(unique_counts_df)

,oa_id,doi,publication_date,created_date,topic_id,topic_name,topic_score,best_date
0,47476822,47476822,114150,22047962,4516,4516,8779542,354937


In [31]:
con = duckdb.connect(db_path)

query = """
    SELECT COUNT(DISTINCT topic_name) 
    FROM my_datasets_topics 
    WHERE topic_score > 0.5
"""

unique_count = con.execute(query).fetchone()[0]

print(f"Unique topics with score > 0.5: {unique_count}")
con.close()

Unique topics with score > 0.5: 4446


In [33]:
con = duckdb.connect(db_path)

query = """
    SELECT 
        topic_name, 
        COUNT(*) AS item_count
    FROM my_datasets_topics
    GROUP BY topic_name
    ORDER BY item_count DESC
"""

topic_counts_df = con.execute(query).df()

display(topic_counts_df)
con.close()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,topic_name,item_count
0,None,32152003
1,Geochemistry and Geologic Mapping,1798104
2,Mycorrhizal Fungi and Plant Interactions,974948
3,Crystallization and Solubility Studies,725063
4,Plasma Diagnostics and Applications,452891
...,...,...
4512,Diffusion Coefficients in Liquids,11
4513,Advanced Scientific and Engineering Studies,10
4514,Globalization and Economic Impact,10
4515,Educational Research and Analysis,9


### Citations

In [6]:
db_path = r"D:\pipeline-data\external\openalex-snapshot\duckdb\openalex_topics_citations.db"

In [10]:
con = duckdb.connect(db_path)
display(con.execute("SELECT* FROM my_datasets_citations LIMIT 5").df())
con.close()

,cited_doi,cited_oa_id,citing_oa_id,citing_doi,citation_date,publication_date,created_date,best_date
0,10.6084/m9.figshare.1510980.v4,W2275684564,W2514643174,10.1002/2015ea000158,2016-07-26,2016-01-01T00:00:00,2016-05-14T13:36:17+00:00,2016-01-01T00:00:00
1,10.15468/bnlkyy,W4395314871,W4396464360,10.15468/dl.hdbkml,2017-08-30,2021-01-01T00:00:00,2016-05-14T14:22:43+00:00,2021-01-01T00:00:00
2,10.6084/m9.figshare.3381109,W4394386444,W4210460784,10.1017/heq.2021.53,2022-02-01,2016-01-01T00:00:00,2016-05-14T14:33:01+00:00,2016-01-01T00:00:00
3,10.7286/v13x84k1,W6977408115,W4385759394,10.3390/biology12081124,2023-08-11,2016-01-01T00:00:00,2016-05-14T15:40:42+00:00,2016-01-01T00:00:00
4,10.6084/m9.figshare.1533109.v4,W4394558733,W4312181754,10.46966/msjar.v3i4.82,2022-12-23,2016-01-01T00:00:00,2016-05-14T19:16:37+00:00,2016-01-01T00:00:00


In [8]:
con = duckdb.connect(db_path)
row_count = con.execute("SELECT count() FROM my_datasets_citations").fetchone()[0]
print(f"Total citations in table: {row_count:,}")
con.close()

Total citations in table: 2,754,955


In [15]:
con = duckdb.connect(db_path)
result = con.execute("SELECT COUNT(DISTINCT cited_doi) FROM my_datasets_citations").fetchone()[0]
print(f"Number of unique cited DOIs: {result:,}")
con.close()

Number of unique cited DOIs: 230,280


In [16]:
con = duckdb.connect(db_path)
result = con.execute("SELECT COUNT(DISTINCT citing_oa_id) FROM my_datasets_citations").fetchone()[0]
print(f"Number of unique citing resources: {result:,}")
con.close()

Number of unique citing resources: 651,237


In [ ]:
con = duckdb.connect(db_path)
result = con.execute("SELECT COUNT(DISTINCT citing_oa_id) FROM my_datasets_citations").fetchone()[0]
print(f"Number of unique citing resources: {result:,}")
con.close()

In [7]:
con = duckdb.connect(db_path)
query = """
    SELECT 
        cited_doi, 
        COUNT(*) AS citation_count
    FROM my_datasets_citations
    WHERE cited_doi IS NOT NULL
    GROUP BY cited_doi
    ORDER BY citation_count DESC
    LIMIT 10
"""
top_10_df = con.execute(query).df()# To see it:
display(top_10_df)
con.close()

,cited_doi,citation_count
0,10.57702/zp44cu3g,25430
1,10.15468/hnhrg3,18598
2,10.57702/kcdhx0zi,18067
3,10.5281/zenodo.5781449,16968
4,10.5281/zenodo.15145663,15746
5,10.57702/o9raffed,15636
6,10.57702/sq75seit,14599
7,10.15468/ib5ypt,11722
8,10.15468/6e8nje,10335
9,10.15468/nc6rxy,9899
